# Phase 1: private FIT structure inspection

Run locally only. Do not attach original files or enter private paths in notebook source.
The notebook prompts for four paths at runtime and reads original ZIP/FIT files without modifying or extracting them.
Reports omit filenames, serial numbers, coordinates, dates and physiological values. Sensor model/protocol metadata is shown, but source classification is deliberately not performed.
Missing external-sensor metadata does not prove wrist HR. A user-assumed armband date is not FIT evidence.
Clear all outputs before saving or committing; never upload the originals or credential files.

## Isolated kernel setup

Run these commands from the repository root in PowerShell. They do not load `.env` or contact Garmin:

```powershell
& 'C:\Python314\python.exe' -m uv venv --python 'C:\Python314\python.exe' .tools\fit-discovery
& 'C:\Python314\python.exe' -m uv pip install --python .tools\fit-discovery\Scripts\python.exe garmin-fit-sdk==21.221.0 ipykernel==7.4.0
```

In VS Code, choose **Select Kernel > Python Environments** and select `.tools\fit-discovery\Scripts\python.exe` (use **Enter interpreter path** if needed).
Check the environment path, not just Python 3.14: the application's `stridelog` kernel does not contain the SDK.
This keeps discovery tools out of the application's dependencies. The SDK license was reviewed before this evaluation.
Then run the cells in order. When prompted, use **Copy as path** on each private ZIP/FIT file and paste it into the runtime input box, not a code cell.
Use a run known to use the armband even if Garmin's activity page omits it.


In [ ]:
import json
from collections.abc import Mapping, Sequence
from datetime import datetime
from itertools import pairwise
from pathlib import Path
from zipfile import BadZipFile, ZipFile

from garmin_fit_sdk import Decoder, Stream

In [ ]:
def _read_fit(path: Path, repo_root: Path) -> bytearray:
    try:
        resolved = path.resolve(strict=True)
    except OSError as error:
        raise ValueError(f"Cannot access sample ({type(error).__name__}).") from None
    if resolved.is_relative_to(repo_root.resolve()):
        raise ValueError("Real samples must be outside the repository.")
    try:
        if resolved.suffix.lower() == ".fit":
            return bytearray(resolved.read_bytes())
        if resolved.suffix.lower() == ".zip":
            with ZipFile(resolved) as archive:
                fits = [
                    item
                    for item in archive.infolist()
                    if not item.is_dir() and Path(item.filename).suffix.lower() == ".fit"
                ]
                if len(fits) != 1:
                    raise ValueError("Expected exactly one FIT member in the ZIP.")
                return bytearray(archive.read(fits[0]))
    except (OSError, BadZipFile, RuntimeError) as error:
        raise ValueError(f"Cannot read sample ({type(error).__name__}).") from None
    raise ValueError("Choose an original ZIP or FIT file.")


def _fields(group: Sequence[Mapping[str, object]]) -> list[str]:
    return sorted({key for message in group for key in message if key != "developer_fields"})


def _inspect_fit(path: Path, repo_root: Path) -> dict[str, object]:
    data = _read_fit(path, repo_root)
    if not Decoder(Stream.from_byte_array(data)).check_integrity():
        raise ValueError("FIT integrity validation failed; no report produced.")
    # Integrity scanning consumes its stream; decode from a fresh one.
    messages, errors = Decoder(Stream.from_byte_array(data)).read()
    if errors:
        raise ValueError("FIT decoding reported errors; no report produced.")
    records = messages.get("record_mesgs", [])
    if not records:
        raise ValueError("No record messages found; check the selected sample.")
    timestamps = [
        timestamp
        for record in records
        if isinstance(timestamp := record.get("timestamp"), datetime)
    ]
    intervals = [(right - left).total_seconds() for left, right in pairwise(timestamps)]
    if not intervals:
        sampling = "insufficient_timestamps"
    elif any(interval <= 0 for interval in intervals):
        sampling = "non_increasing_timestamps"
    elif len(set(intervals)) > 1:
        sampling = "variable_intervals"
    else:
        sampling = "uniform_one_second" if intervals[0] == 1 else "uniform_other_interval"
    devices = messages.get("device_info_mesgs", [])
    models: list[dict[str, str | int]] = []
    for device in devices:
        model: dict[str, str | int] = {}
        for key in (
            "source_type",
            "device_type",
            "ant_device_type",
            "antplus_device_type",
            "ble_device_type",
            "manufacturer",
            "product",
        ):
            value: object = device.get(key)
            if isinstance(value, (str, int)):
                model[key] = value
        models.append(model)
    return {
        "message_types": sorted(messages),
        "record_fields": _fields(records),
        "hr_values_present": any(record.get("heart_rate") is not None for record in records),
        "sampling": sampling,
        "device_info_fields": _fields(devices),
        "sensor_models_not_identifiers": models,
        "lap_fields": _fields(messages.get("lap_mesgs", [])),
        "event_fields": _fields(messages.get("event_mesgs", [])),
        "activity_fields": _fields(messages.get("activity_mesgs", [])),
        "source_classification": "not_performed",
    }

In [ ]:
working = Path.cwd().resolve()
repo_root = next(
    (candidate for candidate in (working, *working.parents) if (candidate / ".git").exists()), None
)
if repo_root is None:
    raise RuntimeError("Run this notebook from the repository or its notebooks folder.")

for label in ("wrist", "strap", "armband", "race"):
    selected = input(f"{label}: private original ZIP/FIT path: ").strip().strip('"')
    if not selected:
        raise ValueError("A private sample path is required.")
    report = _inspect_fit(Path(selected).expanduser(), repo_root)
    print(label)
    print(json.dumps(report, indent=2))

## Interpretation limits

- Share only these projected reports after checking them locally; never paste original data or runtime inputs.
- Sensor models/protocols are not proof that a particular sensor supplied every HR sample. Missing metadata is not proof of wrist HR.
- Sampling labels describe timestamp intervals, not a proven watch recording setting.
- Field presence does not establish units, nullability, elevation quality or physical cadence conversion. Those checks remain separate.
- User-assumed source dates remain private configuration and are not applied by this notebook.
- A decode or integrity error stops inspection rather than returning a success-shaped empty result.
- Clear all outputs before saving; the repository's nbstripout hook is an additional guard, not permission to include private outputs.
